# Public Health Equity—Load & Explore

**Agents for Impact 2026 · Challenge 5**

You are going to build an agent that helps a county health department decide **where to send a
mobile clinic next month**. Not who to treat, not what to prescribe—where to park the van, and
how to explain the choice to the people who live there.

This notebook gets you the data. It does not build the agent. That part is yours, and it is
what you are judged on.

## What you'll have when this finishes

| Table | One row per | What it tells you |
|---|---|---|
| `burden_tracts` | census tract | who is already sick, and who is structurally exposed |
| `care_sites` | facility | where care already exists, with coordinates |
| `shortage_areas` | HRSA designation | where the federal government says there are not enough providers |
| `exposure_tracts` | census tract | air quality, at the scale it actually works |
| `disease_weekly` | week × area | what respiratory illness is doing right now |

## The four lanes, and roughly what each does with the 4.5 hours

**Data lane.** Run this notebook (~8 min), then spend the rest of the morning on the queries
nobody handed you. Distance from each tract to its nearest clinic. Which tracts are in the worst
decile on three measures at once. Whether your county's shortage areas actually overlap its sick
neighbourhoods—they often don't, and that gap is a finding.

**Agent lane.** Start at minute zero, not after lunch. Your differentiator is an *architecture*,
so unlike the other four challenges you cannot bolt it on at the end. Read Section 13 before you
write a line.

**Front-end lane.** You have a deployed endpoint by mid-afternoon. Until then, build against the
tables directly—every question the agent will answer is a query you can run today.

**Story lane.** You get a short pitch deck and a quick demo, and presentation time at this event
is abbreviated. **Rehearse to the clock.** The strongest thing you can say on stage is a number
from your own data that surprised you.

## Before you start

Two things will be asked of you the first time you open Colab Enterprise in a fresh project, and
both are normal rather than errors:

1. Opening Colab Enterprise **prompts you to enable APIs.** Enable them.
2. The Colab Enterprise homepage then shows a **separate "Enable APIs" button** at the top.
   Enable those too. It is a second, distinct step and it is easy to miss.

You do not need to install anything. Everything this notebook uses is already on the runtime.

**If you have never used a notebook before:** each grey block below is a *cell*. Click it and
press Shift+Enter to run it. Run them in order, top to bottom. If one fails, fix it before
moving on—later cells depend on earlier ones.

**If you have:** Runtime → Run all works, and the whole thing takes about eight minutes. Change
`COUNTY_FIPS` in the next cell first.

## 1. Setup

One cell to change, then a cell of plumbing you can ignore.

In [ ]:
# ---- the one thing you change ------------------------------------------------
COUNTY_FIPS = "17031"        # Cook County, Illinois (Chicago)
COUNTY_NAME = "Cook"         # cosmetic, used in printed output only
STATE_ABBR  = "IL"           # must match COUNTY_FIPS, checked below

# Section 3 lists counties we have actually run. Any county works, including ones
# not listed—but read Section 3 before picking, because two things vary by state
# in ways that will surprise you.
# ------------------------------------------------------------------------------

DATASET   = "a4i_health"     # BigQuery dataset created in your own project
LOCATION  = "US"

STATE_FIPS = COUNTY_FIPS[:2]
COUNTY_ONLY = COUNTY_FIPS[2:]

import os, sys, json, time, math, contextlib
import requests
import pandas as pd
from google.cloud import bigquery

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT:
    PROJECT = bigquery.Client().project
bq = bigquery.Client(project=PROJECT, location=LOCATION)

print(f"project     : {PROJECT}")
print(f"county      : {COUNTY_NAME} ({COUNTY_FIPS}), state FIPS {STATE_FIPS}, {STATE_ABBR}")
print(f"dataset     : {PROJECT}.{DATASET}")
print(f"pandas      : {pd.__version__}")

### Why this notebook is timed

Every section below runs inside a `step(...)` block that prints how long it took, and the last
cell totals them. That is not decoration. If you are helping a table at the event and someone
says "it's slow", the timing block turns that into a number, and the number usually points
straight at the culprit—which on this challenge is one 40-second query that we know about and
have decided to keep.

The other thing in the next cell is the validation harness, and it has **two verdicts, not one**,
because a validation suite gets asked two completely different questions:

- **`check()`—did OUR pipeline do its job?** Did the join run, did the numbers parse, are the
  tract IDs eleven characters. A failure means the code is wrong and nothing downstream is
  trustworthy. **Fatal.**
- **`note()`—what did the SOURCE publisher hand us?** We load exactly what CDC and HRSA
  publish, so an implausible value in a federal file is a fact about somebody's data entry, not
  a bug in our code. **It warns, prints the offending rows, and carries on**—escalating to fatal
  only above 1% of rows, because a systemic break looks exactly like a typo until you count it.

That distinction cost a previous challenge in this pack three whole states. Worth knowing before
you write your own checks.

In [ ]:
STEPS, CHECKS = {}, []

@contextlib.contextmanager
def step(name):
    """Time a block and record it. Prints as it goes so a slow cell is visible."""
    t0 = time.time()
    print(f"[{name}] ...")
    try:
        yield
    finally:
        dt = time.time() - t0
        STEPS[name] = round(dt, 1)
        print(f"[{name}] {dt:.1f}s")


def _record(name, verdict, detail):
    """Checks are keyed by NAME, and re-recording one replaces it.

    People re-run cells. A check list that appends every time turns the tally into
    nonsense — you get the same check three times with three different verdicts and no
    way to tell which is current. Last write wins.
    """
    for c in CHECKS:
        if c["check"] == name:
            c["verdict"], c["detail"] = verdict, detail
            return
    CHECKS.append({"check": name, "verdict": verdict, "detail": detail})


def check(name, ok, detail=""):
    """Did OUR pipeline do its job? Fatal when it did not."""
    _record(name, "PASS" if ok else "FAIL", detail)
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"  — {detail}" if detail else ""))
    return bool(ok)


def note(name, bad, total, detail="", limit=0.01, rows=None):
    """What did the SOURCE publisher hand us? Warns, shows the rows, escalates above `limit`.

    total=0 means 'no denominator' — never escalate.
    """
    frac = (bad / total) if total else 0.0
    verdict = "PASS" if bad == 0 else ("FAIL" if (total and frac > limit) else "WARN")
    _record(name, verdict, detail or (f"{bad:,} of {total:,}" if total else f"{bad:,}"))
    print(f"  {verdict}  {name}" + (f"  — {detail}" if detail else
                                    (f"  — {bad:,} of {total:,}" if total else "")))
    if verdict != "PASS" and rows is not None:
        # A check that fails without printing what failed forces a full re-run to diagnose.
        try:
            display(rows.head(10))
        except Exception:
            print(rows)
    return verdict


def safe_str(s, width=None):
    """`.astype(str)` turns None into the literal string 'None'. Zero-pad that and you ship
    '0None' to 150 people. This keeps missing values missing."""
    out = s.astype("string")
    if width:
        out = out.str.strip().str.zfill(width)
    return out.where(out.notna(), None)


def spread(series, label=""):
    """The shape of a distribution, in the terms that decide whether it is usable at all."""
    s = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if s.empty:
        return {}
    p10, p50, p90 = s.quantile([0.10, 0.50, 0.90])
    out = {"n": int(s.size), "min": round(float(s.min()), 3),
           "p10": round(float(p10), 3), "median": round(float(p50), 3),
           "p90": round(float(p90), 3), "max": round(float(s.max()), 3),
           "p90/p10": round(float(p90 / p10), 3) if p10 else None,
           "CV%": round(float(s.std() / s.mean() * 100), 2) if s.mean() else None}
    if label:
        print(f"  {label:<26} " + "  ".join(f"{k}={v}" for k, v in out.items()))
    return out

In [ ]:
UA = {"User-Agent": "A4I2026-challenge5-notebook"}

def get_json(url, params=None, tries=4, timeout=180):
    """One HTTP helper for the whole notebook, with three rules learned the expensive way.

    1. Retry ONLY 429 and 5xx. A 400 is deterministic; retrying it five times just proves
       the same thing five times.
    2. Check status AND content-type before parsing. A service that answers a rate limit
       with an HTML challenge page gives you a JSON parse error that blames the parser.
    3. Back off exponentially, because 150 people are hitting this in the same ten minutes.
    """
    delay = 2.0
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers=UA)
        except Exception as e:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — {type(e).__name__}: {e}")
            time.sleep(delay); delay *= 2; continue
        if r.status_code == 429 or 500 <= r.status_code < 600:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — HTTP {r.status_code} after {tries} tries")
            time.sleep(delay); delay *= 2; continue
        if r.status_code != 200:
            raise RuntimeError(f"{url} — HTTP {r.status_code}: {r.text[:300]}")
        ctype = r.headers.get("content-type", "")
        if "json" not in ctype.lower():
            raise RuntimeError(f"{url} — expected JSON, got {ctype!r}: {r.text[:200]!r}")
        return r.json()
    raise RuntimeError(f"{url} — exhausted retries")


def soda(dataset_id, params, page=50000):
    """CDC's Socrata API, paged. Ask for what you want; never probe past the end."""
    url = f"https://data.cdc.gov/resource/{dataset_id}.json"
    out, offset = [], 0
    while True:
        p = dict(params); p["$limit"] = page; p["$offset"] = offset
        chunk = get_json(url, p)
        out.extend(chunk)
        if len(chunk) < page:
            break
        offset += page
    return pd.DataFrame(out)


DETERMINISTIC_ARCGIS = ("cannot find field", "invalid field", "parsing", "invalid expression",
                        "invalid where", "does not support", "not supported")


class ArcGISTransient(RuntimeError):
    """The service failed in a way a retry might fix. A distinct type so callers can
    decide — arcgis_count degrades to None, everything else gives up honestly."""


def arcgis_query(layer_url, params, tries=3, label="query", passthrough=()):
    """Every ArcGIS /query in this notebook goes through here, for one reason.

    **This service reports failure with HTTP 200 and an error object in the body.**
    get_json retries on status codes, so it is structurally blind to that: a 200 with
    `{"error": {"code": 400, "message": "Unable to complete operation"}}` sails straight
    through and lands in the caller as garbage. Retry has to live at this layer or it
    does not exist for ArcGIS at all.

    Same three outcomes as everywhere else in this notebook:
      * a payload
      * `RuntimeError` for a deterministic error — a wrong field name will still be a
        wrong field name on the fourth try, and hammering the service to prove it is
        rude when 150 people share it
      * `ArcGISTransient` after `tries` attempts with exponential backoff

    `passthrough` is for errors the caller wants to see rather than have raised — the
    paging loop uses it to catch "pagination is not supported" and change strategy.
    """
    delay = 2.0
    last = None
    for attempt in range(tries):
        d = get_json(layer_url + "/query", params)
        err = d.get("error")
        if not err:
            return d
        last = err
        low = str(err).lower()
        if any(k in low for k in passthrough):
            return d
        if any(k in low for k in DETERMINISTIC_ARCGIS):
            raise RuntimeError(f"{layer_url} {label} failed, and retrying will not "
                               f"help — {err}")
        if attempt < tries - 1:
            time.sleep(delay); delay *= 2
    raise ArcGISTransient(f"{layer_url} {label} — {last} (after {tries} tries)")


def arcgis_all(layer_url, where, out_fields, order=None, page=1000):
    """ArcGIS feature service, paged on resultOffset.

    Two rules baked in:
    * NEVER combine returnDistinctValues with paging — that returns HTTP 200 and an empty
      feature list, which looks exactly like 'no data here'.
    * Always pass an explicit outFields list. On one of the layers below that is not a
      performance nicety, it is how we avoid ingesting named individuals.
    """
    rows, offset = [], 0
    while True:
        p = {"where": where, "outFields": out_fields, "returnGeometry": "false",
             "f": "json", "resultOffset": offset, "resultRecordCount": page}
        if order:
            p["orderByFields"] = order
        d = arcgis_query(layer_url, p, label=f"rows at offset {offset}")
        feats = d.get("features", [])
        rows.extend(f["attributes"] for f in feats)
        if len(feats) < page:
            break
        offset += page
    return pd.DataFrame(rows)


def arcgis_points(layer_url, where, out_fields, page=1000):
    """Fetch point features, coping with layers that refuse to paginate.

    These layers publish coordinates as geometry rather than lat/lon columns, so
    `returnGeometry=false` throws away the only thing that makes a facility list useful.

    And pagination support varies BETWEEN LAYERS OF THE SAME SERVICE. Layer 0 of the CMS
    facilities service pages happily; layer 1 answers `resultOffset` with
    400 "Pagination is not supported." So: ask what the layer supports, believe the error
    over the capability flag, and fall back to chunking on OBJECTID — which every ArcGIS
    layer supports, because it is only a WHERE clause.
    """
    meta = get_json(layer_url, {"f": "json"})
    can_page = bool((meta.get("advancedQueryCapabilities") or {}).get("supportsPagination"))
    base = {"where": where, "outFields": out_fields, "returnGeometry": "true", "f": "json"}

    def _rows(d):
        out = []
        for f in d.get("features", []):
            a = dict(f["attributes"])
            g = f.get("geometry") or {}
            a["longitude"], a["latitude"] = g.get("x"), g.get("y")
            out.append(a)
        return out

    if can_page:
        rows, offset = [], 0
        while True:
            d = arcgis_query(layer_url,
                             {**base, "resultOffset": offset, "resultRecordCount": page},
                             label=f"page at offset {offset}", passthrough=("pagination",))
            if d.get("error"):
                can_page = False          # the capability flag lied. Start over unpaged.
                break
            feats = d.get("features", [])
            rows.extend(_rows(d))
            if len(feats) < page:
                return pd.DataFrame(rows)
            offset += page

    d = arcgis_query(layer_url, {"where": where, "returnIdsOnly": "true", "f": "json"},
                     label="id list")
    oid_field = d.get("objectIdFieldName", "OBJECTID")
    oids = d.get("objectIds") or []
    print(f"    (this layer refuses pagination — chunking {len(oids):,} ids on {oid_field})")

    rows = []
    for i in range(0, len(oids), page):
        idlist = ",".join(str(o) for o in oids[i:i + page])
        c = arcgis_query(layer_url, {**base, "where": f"{oid_field} IN ({idlist})"},
                         label=f"id chunk {i // page + 1}")
        rows.extend(_rows(c))
    return pd.DataFrame(rows)


def arcgis_count(layer_url, where, tries=3):
    """Row count, or None. THREE outcomes, not two — and that is the whole point.

    We have now got this wrong in both directions on the same service:

    * `d.get("count", 0)` treats an error as a zero, so an invalid field name and a
      genuinely empty county look identical. That had us believing Houston has no
      shortage designations.
    * Raising on every error kills the cell when HRSA merely hiccups. This layer
      answered `400 Unable to complete operation` once and counted fine on the retry.

    So: a count is a count, a *deterministic* error is a bug and raises, and a
    *transient* one returns **None** after retrying. None is not zero and callers must
    not treat it as zero — it means "the service would not tell us", which is a
    different sentence from "there are none here".

    Same distinction as check() versus note(), one layer down the stack. The retry and
    the deterministic/transient triage live in arcgis_query; the only thing special about
    a count is that we would rather carry on without one than stop the notebook.
    """
    try:
        d = arcgis_query(layer_url, {"where": where, "returnCountOnly": "true", "f": "json"},
                         tries=tries, label="count")
    except ArcGISTransient as e:
        print(f"    (count unavailable: {str(e)[:140]})")
        return None
    if "count" not in d:
        raise RuntimeError(f"{layer_url} count returned no count field — {str(d)[:200]}")
    return int(d["count"])


def load_df(df, table, schema=None):
    """Load a dataframe into BigQuery, replacing whatever was there. --replace everywhere
    makes this notebook safe to re-run and safe to interrupt."""
    ref = f"{PROJECT}.{DATASET}.{table}"
    cfg = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    if schema:
        cfg.schema = schema
    bq.load_table_from_dataframe(df, ref, job_config=cfg).result()
    n = bq.get_table(ref).num_rows
    print(f"  loaded {ref}: {n:,} rows")
    return n

In [ ]:
with step("create dataset"):
    # Test existence with `bq show --dataset PROJECT:DATASET`, never `bq ls -d NAME` — the
    # latter lists datasets in a *project* called NAME and reports "missing" for one that
    # exists. And tolerate "already exists" even behind the check, because checks misfire
    # and two teammates can run this in the same project in the same second.
    ds = bigquery.Dataset(f"{PROJECT}.{DATASET}")
    ds.location = LOCATION
    try:
        bq.get_dataset(ds)
        print(f"  dataset {DATASET} already exists — reusing it")
    except Exception:
        try:
            bq.create_dataset(ds, exists_ok=True)
            print(f"  created dataset {DATASET}")
        except Exception as e:
            if "already exists" not in str(e).lower():
                raise
            print(f"  dataset {DATASET} already exists — reusing it")

## 2. Start with the wrong answer

Before anything else, let's do the obvious thing and watch it fail.

You are siting a mobile clinic. The intuitive move is to send it where the air is worst—so rank
your county's census tracts by average fine-particulate pollution and take the top of the list.
CDC publishes exactly that: **daily PM2.5 estimated at every census tract in the United States**,
from a model EPA built by fusing air-quality monitor readings with an atmospheric simulation.
Tract-level. Daily. Four decimal places.

Run it.

> **This cell takes 30–60 seconds** and it is the slowest thing in the notebook. It is asking
> CDC's servers to average a couple of hundred thousand values for you rather than downloading
> them, which is the only sane way to do it—more on that in Section 6.

In [ ]:
PM25 = "vpk8-vfhm"      # Daily Census Tract-Level PM2.5 Concentrations, 2021-2022

with step("the obvious answer: rank tracts by air quality"):
    # Note there is no `count(distinct date)` here, tempting though it is. This endpoint
    # accepts it, returns HTTP 200, and gives you a wrong number — Section 6 measures the
    # grain a way that actually works.
    pm = soda(PM25, {
        "$select": ("ctfips, avg(ds_pm_pred) as mean_pm, max(ds_pm_pred) as max_pm, "
                    "count(*) as n_rows"),
        "$where": f"ctfips like '{COUNTY_FIPS}%'",
        "$group": "ctfips",
    }, page=5000)

    for c in ("mean_pm", "max_pm", "n_rows"):
        pm[c] = pd.to_numeric(pm[c], errors="coerce")

    print(f"  tracts: {len(pm):,}")
    worst = pm.sort_values("mean_pm", ascending=False).head(5)
    best  = pm.sort_values("mean_pm").head(5)
    print("\n  The five worst tracts in the county:")
    for _, r in worst.iterrows():
        print(f"    {r.ctfips}   {r.mean_pm:.3f} ug/m3")
    print("\n  The five best tracts in the county:")
    for _, r in best.iterrows():
        print(f"    {r.ctfips}   {r.mean_pm:.3f} ug/m3")

    s = spread(pm["mean_pm"], "mean PM2.5 across tracts")

### Look at the two lists before you read on

The worst tract and the best tract in the county. What separates them?

In Cook County it is **8.91 against 10.14**—about a microgram, or fourteen percent, between the
single most polluted census tract in Chicago and the single least polluted one. The tenth and
ninetieth percentiles are separated by **five percent**.

If you sent the van to the top tract instead of the median tract, you would be responding to a
difference of roughly two percent in average exposure.

Now run the same measurement on a different column, for the same tracts.

In [ ]:
PLACES = "yjkw-uj5s"    # PLACES: Census Tract Data (GIS Friendly Format), 2025 release

with step("the same tracts, a different column"):
    burden = soda(PLACES, {
        "$select": ("tractfips, totalpopulation, geolocation, "
                    "access2_crudeprev, casthma_crudeprev, copd_crudeprev, "
                    "diabetes_crudeprev, bphigh_crudeprev, checkup_crudeprev, "
                    "mobility_crudeprev_, selfcare_crudeprev, indeplive_crudeprev, "
                    "disability_crudeprev"),
        "$where": f"countyfips='{COUNTY_FIPS}'",
    }, page=5000)
    print(f"  tracts: {len(burden):,}")

    print()
    spread(pm["mean_pm"],                   "PM2.5 (average)")
    spread(burden["casthma_crudeprev"],     "asthma prevalence %")
    spread(burden["copd_crudeprev"],        "COPD prevalence %")
    spread(burden["access2_crudeprev"],     "uninsured 18-64 %")

### What just happened

Same county. Same census tracts. Four columns.

| Column | p90 / p10 in Cook County |
|---|---|
| Average PM2.5 | **1.05×** |
| Asthma prevalence | 1.48× |
| COPD prevalence | 3.18× |
| **Uninsured rate** | **4.77×** |

**The uninsured rate varies roughly ninety times more across these neighbourhoods than the air
does.** We measured this in four counties before writing this notebook—Cook, Travis, Harris
County (Houston, with its ship channel and refineries), and the Bronx. Houston, which ought to
be the strongest pollution-gradient case in the United States, separates its cleanest and
dirtiest decile by nine percent. **The Bronx—which has one of the most documented asthma
disparities in the country—separates by one and a third percent.**

So why does a dataset published *per census tract*, with four decimal places and 730 rows for
every tract, fail to tell one neighbourhood from another?

**Because it was never a tract-level measurement.** EPA's model runs on a **12-kilometre grid**,
then reports the value at each tract's centre point. A 12 km cell covers dozens of tracts, and
they all inherit the same number. The Bronx is about 10 km across. It is roughly one cell.

**Precision is not resolution.** The identifier is real, the decimals are real, the daily
frequency is real, and the spatial detail is not there. Nothing in the file says so, no query
fails, and every ranking you build on it will look perfectly reasonable.

That is the trap this notebook exists to walk you out of, and it sets up the actual shape of the
challenge:

- **Air quality tells you about counties and about bad days.** It is real at that scale—Houston
  averages 10.4 against the Bronx's 8.4, and the Bronx's worst day hit 57. Section 6 uses it for
  exactly that and no more.
- **Who is sick and who cannot reach care tells you about neighbourhoods.** Sections 4, 5 and 7.
- **Your agent will have to reason across both**, and say which claim came from which scale.
  That is not a nuisance. It is the job.

## 3. Now pick your county

You are building for **one county health department**. Not a state, not the nation—a department
with one van, a budget, and a catchment they already know by name.

Any county in the United States works. But two things vary by state in ways that will cost you
an hour if you meet them by surprise, so read this before you commit.

### The four we have actually run

Every number here was measured, not estimated. **The last three columns are the ones that will
surprise you**, and they are the reason this table exists rather than a list of city names.

| County | Tracts | Population | Uninsured 18-64 p10→p90 | Asthma p10→p90 | Tracts with a real safety-net gap | Worst km to a safety-net clinic | Shortage **areas** desig/all | Shortage **facilities** | Transport measure |
|---|---:|---:|---|---|---:|---:|---:|---:|:-:|
| **Cook IL** (Chicago) | 1,328 | 5,275,505 | 5.1% → 24.3% | 8.8% → 13.0% | **215** | 20.7 | **498 / 934** | 32 | ✅ |
| **Harris TX** (Houston) | 1,110 | 4,731,109 | **8.9% → 40.8%** | 8.3% → 10.7% | **263** | 22.8 | **0** | 15 | ❌ |
| **Travis TX** (Austin) | 289 | 1,290,185 | 6.4% → 28.2% | 8.8% → 10.1% | 62 | 21.7 | **0** | 2 | ❌ |
| **Bronx NY** | 347 | 1,472,508 | 7.5% → 23.0% | **10.5% → 13.4%** | 19 | 4.8 | 338 / 338 | 8 | ✅ |

**The last two columns are not two versions of the same thing, and confusing them is the single
easiest mistake to make in this dataset.** A shortage *area* says a neighbourhood is underserved.
A shortage *facility* says a clinic serves an underserved population—it marks where care already
**is**. Section 7 is blunt about why that matters.

Read it as a menu of *stories*, not a ranking.

**Cook is the default** because it is the only one with everything: shortage-area designations,
the transportation measure, the widest asthma spread, and 215 neighbourhoods where the nearest
clinic that treats the uninsured is meaningfully further than the nearest clinic of any kind.
It also demonstrates a trap this notebook warns about, live: **498 of its 934 designations are
`Proposed For Withdrawal`.** Skip the status filter and you nearly double your shortage areas
with ones on their way out.

**Harris has the strongest raw inequality**—forty percent uninsured at the ninetieth
percentile—and the most gap tracts of any county we measured. It has no HPSA data at all.

**Travis is the smallest and fastest to iterate on**, which makes it a good development county
and a thin demo.

**The Bronx has the highest asthma floor in the country and almost no distances.** Median tract
is 510 metres from care. That makes it the sharpest test of whether your agent understands what
it is looking at, and the weakest case for a mobile clinic.

> **Two of the four have no shortage-area data whatsoever.** Travis and Harris both return zero
> primary-care HPSA *components* while Texas has 1,234 statewide, and it is not a filter
> error—the identical query returns 934 for Cook. Some counties simply have no area designation.
> `shortage_areas` ships empty, the notebook warns rather than fails, and if you picked one of
> those counties you have lost `HPSA_SCORE` as a *neighbourhood* signal, which is our only
> travel-burden measure. **Check that column before you commit to a county.**

> **Why a county, and not a city or a state?** Because a county health department is a real
> organisation with a real mobile clinic, and the county is the level at which somebody actually
> decides where the van goes on Tuesday. It is also the finest level at which the current
> disease-activity data resolves at all—Section 8 is blunt about that.

### The two things that vary by state

**1. Seven measures are published for only 40 states.** CDC PLACES splits into core health
measures, which are nearly national, and *health-related social needs*—lack of transportation,
food insecurity, utility shut-off threat, social isolation—which are published for **40 states
and not the other eleven**.

**Missing: Colorado, Florida, Kentucky, Oregon, Pennsylvania, South Dakota, Tennessee, Texas,
Vermont, Washington, Wyoming.**

That includes Texas, so if you follow the default county you will not have `lacktrpt`. This is
not a bug in this notebook and not something you can work around—the rows do not exist. The
next cell prints exactly which measures your county has, and you should look at it. Note also
that even `casthma`, one of the core measures, covers 49 states rather than all 51.

**2. Tract identifiers lose their leading zero in seven states.** Any state with a FIPS code
below 10—Alabama 01, Alaska 02, Arizona 04, Arkansas 05, **California 06**, Colorado 08,
Connecticut 09—has tract IDs that some federal products store as numbers, stripping the zero.
An eleven-character ID becomes ten, joins return zero rows, and nothing raises. Every join in
this notebook pads to eleven characters. If you add a data source of your own, do the same.

**Pick a county whose story you can tell in ninety seconds on stage.** A judge remembers "the
two tracts on the east side where a third of working-age adults have no insurance and the
nearest clinic is eleven kilometres away." Nobody remembers a heat map.

In [ ]:
with step("what your county actually has"):
    # value_counts() before you design anything around a categorical. A measure that is
    # absent for your state returns zero rows, and zero rows is indistinguishable from
    # "this feature does not exist anywhere" unless you go and look.
    HRSN = ["LACKTRPT", "FOODINSECU", "SHUTUTILITY", "EMOTIONSPT",
            "LONELINESS", "HOUSINSECU", "FOODSTAMP"]
    CORE = ["CASTHMA", "COPD", "DIABETES", "BPHIGH", "ACCESS2", "CHECKUP",
            "MOBILITY", "SELFCARE", "INDEPLIVE", "DISABILITY"]
    inlist = ",".join(f"'{m}'" for m in HRSN + CORE)
    got = get_json("https://data.cdc.gov/resource/cwsq-ngmh.json",
                   {"$select": "measureid, count(*) as n",
                    "$where": f"stateabbr='{STATE_ABBR}' AND measureid in ({inlist})",
                    "$group": "measureid", "$limit": 100})
    present = {g["measureid"] for g in got}
    missing = [m for m in HRSN + CORE if m not in present]

    print(f"  present in {STATE_ABBR}: {len(present)} of {len(HRSN)+len(CORE)}")
    if missing:
        print(f"  NOT PUBLISHED for {STATE_ABBR}: {', '.join(missing)}")
        print("  Those columns will be absent below. That is the publisher, not this notebook.")
    else:
        print("  every measure we use is published for your state")

    # And the leading-zero class, stated rather than assumed.
    if int(STATE_FIPS) < 10:
        print(f"\n  NOTE: state FIPS {STATE_FIPS} is below 10. Your tract IDs are in the class")
        print("  where the leading zero gets stripped by some publishers. Every join below")
        print("  pads to 11 characters; do the same for anything you add.")

## 4. Who is already sick—CDC PLACES

This is the layer that actually varies between neighbourhoods, so it is the one worth
understanding properly.

**PLACES gives you model-based small-area estimates of health conditions for every census tract
in the country.** Forty measures. We pulled twelve of them in Section 2 and we are going to keep
all twelve, because which ones matter depends on the story you choose to tell.

The four groups, and why each earns its place in a *mobile clinic siting* problem:

| Group | Measures | Why a van cares |
|---|---|---|
| **Respiratory** | `casthma`, `copd` | The conditions air quality actually acts on, and the ones a screening clinic can catch early |
| **Chronic, screenable** | `diabetes`, `bphigh` | A blood-pressure cuff and a finger-prick find these. This is what preventive vans are *for* |
| **Access** | `access2` (uninsured 18-64), `checkup` (routine visit in the past year) | Somebody with no insurance and no checkup in a year is the population you are driving toward |
| **Cannot come to you** | `mobility`, `selfcare`, `indeplive`, `disability` | If they cannot walk to the corner, a van two neighbourhoods away is not access |

### Three things to know before you rank anything on these

**They are estimates, not counts.** PLACES is built by fitting a model to national survey data
and projecting it onto small areas. The value for your tract is a *prediction about people like
the people who live there*, not a tally of anybody. It cannot tell you that a specific person
has asthma, and it smooths toward the mean—so the genuinely extreme tract is probably more
extreme than the number says.

**They come with a confidence interval and you should look at it.** Every measure has a
`_crude95ci` twin. We are not loading those by default, but if two tracts differ by less than
their intervals overlap, your ranking is decorative. Loading them is a two-line change and a
strong add-on.

**One column name is broken and it is not a typo on our side.** Every measure follows the
pattern `<name>_crudeprev`—except mobility, which is **`mobility_crudeprev_`, with a trailing
underscore.** Build your column names in a loop and you get a `KeyError` on exactly one of the
four disability measures. We handle it explicitly below rather than papering over it, because
you will meet the same class of thing in whatever you add next.

In [ ]:
with step("shape the burden layer"):
    # Rename to something you can type, and fix the publisher's trailing underscore.
    RENAME = {
        "tractfips": "geo_id",
        "totalpopulation": "total_pop",
        "access2_crudeprev": "pct_uninsured",
        "casthma_crudeprev": "pct_asthma",
        "copd_crudeprev": "pct_copd",
        "diabetes_crudeprev": "pct_diabetes",
        "bphigh_crudeprev": "pct_high_bp",
        "checkup_crudeprev": "pct_checkup",
        "mobility_crudeprev_": "pct_mobility_difficulty",   # note the trailing underscore
        "selfcare_crudeprev": "pct_selfcare_difficulty",
        "indeplive_crudeprev": "pct_indep_living_difficulty",
        "disability_crudeprev": "pct_any_disability",
    }
    missing_cols = [c for c in RENAME if c not in burden.columns]
    if missing_cols:
        print(f"  columns absent from the API response: {missing_cols}")

    b = burden.rename(columns=RENAME).copy()

    # Coordinates come from PLACES itself, and this is worth a sentence. The obvious source
    # for tract coordinates is BigQuery's geo_census_tracts, but that table is 2010-vintage
    # while PLACES, SVI and ACS are all 2020 — and the boundaries were redrawn, so anchoring
    # on it silently discards about a fifth of a county. PLACES ships a `geolocation` GeoJSON
    # point per tract, same vintage as its own data. Free, exact, no drift.
    def _lonlat(g):
        try:
            return g["coordinates"][0], g["coordinates"][1]
        except Exception:
            return None, None
    lonlat = b["geolocation"].apply(_lonlat)
    b["longitude"] = [x[0] for x in lonlat]
    b["latitude"]  = [x[1] for x in lonlat]
    b = b.drop(columns=["geolocation"])

    b["geo_id"] = safe_str(b["geo_id"], width=11)
    for c in b.columns:
        if c.startswith("pct_") or c in ("total_pop", "longitude", "latitude"):
            b[c] = pd.to_numeric(b[c], errors="coerce")

    print(f"  tracts: {len(b):,}   columns: {len(b.columns)}")
    print(f"  with coordinates: {b['latitude'].notna().sum():,}")
    print(f"  population in county: {int(b['total_pop'].sum()):,}")

## 5. Who is structurally exposed—CDC/ATSDR Social Vulnerability Index

PLACES tells you who is sick. SVI tells you who will have trouble doing anything about it, and
it gives us two columns nothing else does: **households with no vehicle**, and **disability at
tract level**.

It also gives us the pre-computed **percentile rank** for every variable—`EPL_*`, on a 0-to-1
scale against every tract in the country. For a "rank the neighbourhoods" problem that is
usually the column you actually want, and most teams miss it and re-derive it badly.

### The part we refuse to use, and why it matters more than the rule

SVI has four themes. **We use Themes 1, 2 and 4. We do not use Theme 3.**

Theme 3 is racial and ethnic minority status. This programme prohibits race and ethnicity as a
**model input**, and requires them instead as a **post-hoc audit of the output**. The reasoning
is worth having straight before a judge asks, because the rule on its own is easy to
misunderstand:

- **Race genuinely does correlate with these outcomes.** Do not claim otherwise; anyone who
  knows the literature will correct you, and they will be right.
- **But it is a proxy for things we can measure directly.** The causal variables here are
  structural and physical—insurance coverage, vehicle access, distance to a clinic, whether the
  neighbourhood has a provider at all. Race is a cruder measurement of something we already have
  a better instrument for.
- **Removing the column does not remove the bias.** Correlated proxies survive. This is
  "fairness through unawareness" and it does not work. The remedy is auditing what your agent
  *recommends*, not deleting an input.

This is consistent with Google's own published responsible-AI guidance, so it is not a rule we
invented for this event.

**Practical consequence: `RPL_THEMES`, the overall SVI score, is contaminated.** It has Theme 3
baked into it and there is no column to drop. Use `RPL_THEME1`, `RPL_THEME2` and `RPL_THEME4`
individually and leave the composite alone. The next cell asks the service for its field list
and requests a named subset—`RPL_THEMES` is deliberately absent from that list, not filtered out
afterwards.

**On this challenge that reasoning is not a side note.** Your agent produces a ranked list of
neighbourhoods to send a clinic to. Section 13 asks you to audit that list against the
demographics you were forbidden to rank on. That audit is the single strongest argument for the
architecture you are about to build, and it is why this section is here rather than in an
appendix.

**One more trap: `-999` is the missing-data sentinel.** Leave it in and your averages are
spectacular.

In [ ]:
SVI = ("https://onemap.cdc.gov/onemapservices/rest/services/SVI/"
       "CDC_ATSDR_Social_Vulnerability_Index_2022_USA/MapServer/2")

with step("pull CDC SVI"):
    # Ask the service what it has rather than trusting a list we typed months ago.
    meta = get_json(SVI, {"f": "json"})
    available = {f["name"] for f in meta.get("fields", [])}
    wanted = ["FIPS", "E_TOTPOP", "EP_NOVEH", "EPL_NOVEH", "EP_DISABL", "EP_AGE65",
              "EP_AGE17", "EP_UNINSUR", "EPL_UNINSUR", "EP_LIMENG", "EP_MOBILE",
              "EP_GROUPQ", "EP_NOINT", "RPL_THEME1", "RPL_THEME2", "RPL_THEME4"]
    use = [f for f in wanted if f in available]
    absent = [f for f in wanted if f not in available]
    print(f"  requesting {len(use)} fields; not present: {absent or 'none'}")
    print(f"  RPL_THEMES exists on this service: {'RPL_THEMES' in available}"
          f" — we are not requesting it")

    svi = arcgis_all(SVI, f"FIPS LIKE '{COUNTY_FIPS}%'", ",".join(use), order="FIPS")
    print(f"  tracts: {len(svi):,}")

    n_sentinel_total = 0
    for c in use:
        if c == "FIPS":
            continue
        svi[c] = pd.to_numeric(svi[c], errors="coerce")
        n = int((svi[c] == -999).sum())
        if n:
            n_sentinel_total += n
            print(f"    {c}: nulling {n} rows carrying the -999 sentinel")
        svi.loc[svi[c] == -999, c] = None
    if not n_sentinel_total:
        print("    no -999 sentinels in this county")

    svi["FIPS"] = safe_str(svi["FIPS"], width=11)
    svi = svi.rename(columns={
        "FIPS": "geo_id", "E_TOTPOP": "svi_total_pop",
        "EP_NOVEH": "pct_no_vehicle", "EPL_NOVEH": "pctile_no_vehicle",
        "EP_DISABL": "pct_disability_svi", "EP_AGE65": "pct_age_65_plus",
        "EP_AGE17": "pct_age_17_under", "EP_UNINSUR": "pct_uninsured_all_ages",
        "EPL_UNINSUR": "pctile_uninsured", "EP_LIMENG": "pct_limited_english",
        "EP_MOBILE": "pct_mobile_homes", "EP_GROUPQ": "pct_group_quarters",
        "EP_NOINT": "pct_no_internet", "RPL_THEME1": "svi_socioeconomic",
        "RPL_THEME2": "svi_household", "RPL_THEME4": "svi_housing_transport"})

In [ ]:
with step("assemble burden_tracts"):
    # Report two counts around every join. A join that ate your data looks exactly like
    # "no data here"; one count cannot tell them apart and a ratio can.
    before = len(b)
    v = b.merge(svi, on="geo_id", how="left")
    matched = int(v["pct_no_vehicle"].notna().sum())
    print(f"  tracts before SVI join: {before:,}   matched after: {matched:,} "
          f"({matched/before:.0%})")

    # Two independent uninsured measures, deliberately kept side by side. They measure
    # different universes and they will not agree — see the markdown below.
    v["uninsured_gap"] = v["pct_uninsured"] - v["pct_uninsured_all_ages"]
    n_tracts = load_df(v, "burden_tracts")

### Two uninsured columns, and they disagree on purpose

`burden_tracts` now carries **`pct_uninsured`** from PLACES and **`pct_uninsured_all_ages`** from
SVI, plus the difference between them.

They are not redundant and they are not inconsistent. PLACES measures **adults aged 18 to 64**.
SVI measures the **whole civilian non-institutionalised population**. Because Medicare covers
almost everyone over 65, PLACES will read *higher* than SVI wherever a tract skews elderly—and
the gap between them is, roughly, a signal about age structure.

**Pick one as your headline number and say which.** A team that reports both without noticing
they measure different populations has an inconsistency in its own output that a judge will find
in about fifteen seconds. A team that reports the gap *deliberately*, as a proxy for who is
already covered by Medicare, has done something interesting.

This is the general shape of the whole challenge: nearly every number here is real, and nearly
every one of them measures something slightly different from what its name suggests.

## 6. The air, at the scale it actually works

Section 2 showed that average PM2.5 cannot tell one neighbourhood from another. That is not a
reason to throw the layer away—it is a reason to use it for the two things it *is* good at.

**Between counties it is real.** Median tract mean: Harris 10.36, Cook 9.83, Travis 8.92,
Bronx 8.37. That is a 24% spread and it reflects genuine differences in industry, traffic and
climate. If your agent is ever asked "is this county's air a problem at all", this answers it.

**Bad days are real.** Travis's worst day averaged 27.2 µg/m³ against a normal 8.9. The Bronx
touched 57. Those are wildfire smoke, winter inversions, and holiday fireworks—and they are the
days a respiratory outreach programme actually cares about.

**And the tail discriminates where the mean does not.** Rank tracts by average and you get a
1.04× spread. Rank them by *number of days above a threshold* and it opens up:

| Threshold | Travis, days per tract in 2021 | spread |
|---|---|---|
| > 9 µg/m³ | 117 – 154 | 1.32× |
| > 12 | 56 – 78 | 1.39× |
| > 15 | 22 – 37 | 1.68× |
| **> 20** | **3 – 9** | **3.0×** |
| > 25 | 1 – 3, in 269 of 290 tracts | 3.0× |
| > 35 | none at all |—|

**Read that honestly, though.** A sampled daily estimate came back as 3.65 with a standard error
of **1.32**. Near a threshold, whether a day counts is partly the model's residual rather than
the air. A 3× spread on "days over 20" may be a 3× spread on where the downscaler is least
certain. `ds_pm_stdd` is in the data and checking your ranking against it is one of the better
add-ons on this challenge. Note also that nine days and three days is a real ratio and a small
difference; be careful what weight you put on it.

### The four traps in this file, all of which fail silently

**1. `latitude` and `longitude` are transposed. In every row, in every one of these datasets.**

```
{"ctfips": "48453000101", "latitude": "-97.75319", "longitude": "30.32334", ...}
```

Austin is at 30.3° north, 97.8° west. The field named `latitude` holds the longitude. Plot these
as published and your county appears in the Indian Ocean, off the coast of Somalia—no error, no
warning, just a map that is wrong. **We ignore both columns and join on `ctfips`**, which is
also faster. Coordinates come from PLACES.

**2. Sibling datasets type their keys differently.** The PM2.5 files store `ctfips` as text; the
2016–2020 ozone file stores it as a **number**. A `LIKE '48453%'` filter that works perfectly on
one returns nothing on the other—HTTP 200, empty list, no complaint. Whichever you use, print
the row count and look at it.

**3. `date` is text in SAS format**, like `01JAN2021`, not ISO-8601. Socrata date filtering does
not work on it. Filter on `year` and parse with `%d%b%Y`. And do not sort those strings—`01APR`
sorts before `01JAN`, so an alphabetical "date range" is a statement about the alphabet.

**4. `count(distinct ...)` is accepted and returns a wrong answer.** Not an error—HTTP 200 and a
number that is not the count of distinct values. This is worse than an unsupported function,
because you get a plausible figure and no reason to doubt it. The next cell measures the grain
by pulling one tract in full and counting locally, which is the only way we found that works.

### Two things about this file that are not what the label says

**It is one year, not two.** The dataset is titled *"Daily Census Tract-Level PM2.5
Concentrations, 2021-2022"*. Its `year` column contains only **2021**, and the true date range
is 2021-01-01 to 2021-12-31. Verified 2026-08-10. If you need 2022 you will not find it here.

**Every tract-day is published twice, with identical values.** 730 rows per tract, 365 distinct
dates, both rows for 01JAN2021 reading `3.651 / 1.3225`. Harmless for `avg()`. It doubles every
`COUNT`, which is why the next cell divides the exceedance counts by the multiplier it measures
rather than by a 2 somebody typed in.

That pair is the whole lesson of this section in miniature: **the row count is not the day count,
and the title is not the contents.** Measure both.

### Never download the daily rows

One county-year is roughly **106,000 tract-days**, published as 212,000 rows. A large county is
millions. Ask CDC's servers to aggregate for you—`$select` with `avg()` and `$group=ctfips`—and
that becomes 290 rows in about forty seconds. Section 2 already ran that query; this section
characterises the grain, then adds exceedance counts that are corrected for it.

In [ ]:
with step("characterise the grain before counting anything"):
    # `count(*)` said 730 rows per tract, which reads like two years of daily values. That
    # is an inference and this notebook does not ship inferences. Pull ONE tract in full
    # and count locally — nunique() on the natural key, with an instrument that works.
    _t = pm["ctfips"].iloc[0]
    one = soda(PM25, {"$select": "year, date, ds_pm_pred, ds_pm_stdd",
                      "$where": f"ctfips='{_t}'"}, page=50000)
    one["ds_pm_pred"] = pd.to_numeric(one["ds_pm_pred"], errors="coerce")
    one["parsed"] = pd.to_datetime(one["date"], format="%d%b%Y", errors="coerce")

    print(f"  sample tract {_t}: {len(one):,} rows, {one['date'].nunique():,} distinct dates")
    print(f"  true date range: {one['parsed'].min():%Y-%m-%d} to {one['parsed'].max():%Y-%m-%d}")
    print(f"  years present  : {sorted(one['year'].unique().tolist())}")

    per_date = one.groupby(["year", "date"]).size()
    uniform = (per_date.nunique() == 1)
    MULT = int(per_date.iloc[0]) if uniform else 1
    pair = one[one["date"] == one["date"].iloc[0]]
    identical = pair["ds_pm_pred"].nunique() == 1

    print(f"  rows per tract-day: "
          f"{MULT if uniform else sorted(per_date.unique().tolist())}"
          f"   identical values: {identical}")

    note("PM2.5 publishes one row per tract per day", 0 if MULT == 1 else 1, 0,
         f"each tract-day is published {MULT}x"
         + (" with identical values — harmless for avg(), doubles every COUNT"
            if identical else " with DIFFERING values — investigate before counting"))
    note("the file covers the years its title claims", 0 if len(one["year"].unique()) > 1 else 1,
         0, f"title says 2021-2022; the data contains {sorted(one['year'].unique().tolist())}")

with step("air quality: exceedance"):
    # The metric that actually separates tracts — corrected by the multiplier we just
    # measured, rather than by a number somebody typed in.
    THRESHOLDS = [12, 20, 25, 35]
    ex = pm[["ctfips", "mean_pm", "max_pm"]].copy()
    for thr in THRESHOLDS:
        col = f"days_over_{thr}"
        d = soda(PM25, {"$select": "ctfips, count(*) as rows_over",
                        "$where": f"ctfips like '{COUNTY_FIPS}%' AND ds_pm_pred > {thr}",
                        "$group": "ctfips"}, page=5000)
        if len(d):
            d["rows_over"] = pd.to_numeric(d["rows_over"], errors="coerce")
            d[col] = (d["rows_over"] / MULT).round().astype(int)
            ex = ex.merge(d[["ctfips", col]], on="ctfips", how="left")
        else:
            ex[col] = 0
        ex[col] = ex[col].fillna(0).astype(int)
        print(f"    > {thr:>2} ug/m3: {int((ex[col] > 0).sum()):,} of {len(ex):,} "
              f"tracts ever exceed it")

    print(f"\n  (day counts divided by the measured {MULT}x publication factor)")
    spread(ex["mean_pm"], "mean PM2.5")
    for thr in THRESHOLDS:
        if ex[f"days_over_{thr}"].max() > 0:
            spread(ex[f"days_over_{thr}"], f"days over {thr}")

    ex = ex.rename(columns={"ctfips": "geo_id"})
    ex["geo_id"] = safe_str(ex["geo_id"], width=11)
    n_exposure = load_df(ex, "exposure_tracts")

### A note on the two air files, if you use ozone as well

We ship PM2.5 only, and that is a decision rather than an oversight. The ozone product covers
2016–2020, PM2.5 covers 2021–2022, and **their tract sets are not the same**. In Travis County
the PM2.5 file has 290 tracts and the ozone file has 420, with all 290 present in both and 130
extra on the ozone side. Join them without thinking and you silently drop rows.

If you want ozone—and for a summer respiratory story you might—use dataset `hf2a-3ebq`, remember
its keys are numeric so `LIKE` will not work, join on the **intersection**, and report the row
count on both sides. That is a genuinely good add-on and it is about fifteen lines.

## 7. Where care already is

A mobile clinic is worth sending where care *is not*. So we need the opposite map: every
federally-recognised care site in the state, with coordinates, placed into your county.

Four layers, all from HRSA's GIS portal, all points, all licence-clean:

| Layer | What it is |
|---|---|
| Hospitals | CMS-certified, from the Provider of Services extract |
| Critical Access Hospitals | Small rural hospitals with a special designation |
| Federally Qualified Health Centers | The safety net—sliding-scale primary care |
| Rural Health Clinics | The other half of the rural safety net |

Plus **HRSA Health Center service delivery sites**, which is a different list built from the
grant side rather than the certification side, and carries something none of the others do:
`TOT_OPER_HR_PER_WEEK`. A clinic open eight hours a week and one open sixty are not the same
supply, and only this layer knows the difference.

### The rule that is not optional on this section

**The health-center layer contains named individuals.** Three columns—`FQHC_ADMIN_CONTACT_NM`,
`FQHC_ADMIN_CONTACT_EMAIL`, `FQHC_ADMIN_CONTACT_PHONE_NUM`—hold a real person's name, email
address and phone number for each site.

Individual-level personal data is an automatic rejection at this event. The only reliable
defence is **never requesting the columns**: an explicit `outFields` allow-list, never `*`. That
is written into the helper and it is a written rule, not a convention. If you extend this
section, keep it.

### A field that looks like the answer and is empty

The Rural Health Clinic layer publishes `PHY_FTE_CT`, `NURSE_PRACT_FTE_CT`, `PHY_ASSIST_FTE_CT`
and `OTHR_PERSONNEL_FTE_CT`—staffed clinical capacity per clinic. That would be far better than
a dot on a map, because two clinics are not equivalent supply if one has two physicians and the
other has a part-time nurse practitioner.

**They are empty. Zero of 387 Texas rows carry a value in any of the four.**

We wrote them into our own research notes as the best find in the dataset before checking, which
is exactly the mistake this notebook keeps warning you about. A field that exists, is named
correctly, and sits right beside the real data is not the same as a field with data in it.
**Rank candidate columns by answered-value count before you choose one.** The cell below prints
the count so you can see it rather than take our word for it.

In [ ]:
HRSA = "https://gisportal.hrsa.gov/server/rest/services"
CMSF = f"{HRSA}/HealthCareFacilities/CMSApprovedFacilities_FS/MapServer"
HCS  = f"{HRSA}/HealthCareFacilities/PrimaryHealthCareFacilities_FS/MapServer/0"

with step("pull care sites"):
    CMS_FIELDS = ("CMS_PROVIDER_NUM,FACILITY_NM,CMS_PROVIDER_ADDRESS,CMS_PROVIDER_CITY,"
                  "CMS_PROVIDER_STATE_ABBR,CMS_PROVIDER_ZIP_CD,CMS_PROVIDER_CAT_DESC,"
                  "CMS_PROVIDER_CAT_SUB_TYP_DESC")
    LAYERS = {0: "Hospital", 1: "Critical Access Hospital",
              2: "FQHC", 3: "Rural Health Clinic"}

    frames = []
    for lid, label in LAYERS.items():
        url = f"{CMSF}/{lid}"
        where = f"CMS_PROVIDER_STATE_ABBR='{STATE_ABBR}'"
        # `returnCountOnly` on this service is INTERMITTENTLY unreliable. Across two runs
        # it returned 0 for Critical Access Hospitals once and for Rural Health Clinics
        # once, on layers that counted correctly the other time. So the fallback is not a
        # per-layer workaround, it is load-bearing — and it must use the SAME check name
        # in both branches, or a flaky count leaves two entries in the tally.
        expected = arcgis_count(url, where)          # may be None — see the helper
        d = arcgis_points(url, where, CMS_FIELDS)
        if len(d):
            d["site_kind"] = label
        said = f"{expected:,}" if expected is not None else "  n/a"
        print(f"  {label:<26} service said {said:>6}   got {len(d):>5,}")
        nm = f"{label}: got the rows the service said exist"
        if expected is None:
            check(nm, len(d) > 0,
                  f"{len(d):,} rows; the service would not report a count after 3 tries, "
                  f"so there is nothing to compare against")
        elif expected > 0:
            check(nm, len(d) == expected, f"{len(d):,} of {expected:,}")
        else:
            check(nm, len(d) == 0, "the service reports zero and we loaded zero")
        frames.append(d)

    # The FTE columns, measured rather than assumed.
    rhc = arcgis_query(f"{CMSF}/3",
                       {"where": f"CMS_PROVIDER_STATE_ABBR='{STATE_ABBR}'",
                        "outFields": "PHY_FTE_CT,NURSE_PRACT_FTE_CT,PHY_ASSIST_FTE_CT,"
                                     "OTHR_PERSONNEL_FTE_CT",
                        "returnGeometry": "false", "f": "json"},
                       label="RHC capacity columns")
    rf = pd.DataFrame([f["attributes"] for f in rhc.get("features", [])])
    if len(rf):
        print("\n  Rural Health Clinic capacity columns, answered-value count:")
        for c in rf.columns:
            print(f"    {c:<24} {int(rf[c].notna().sum()):>4} of {len(rf):>4} non-null")

In [ ]:
with step("pull HRSA health center sites"):
    # EXPLICIT allow-list. Never '*' on this layer — see the markdown above.
    HC_FIELDS = ("SITE_NM,SITE_ADDRESS,SITE_CITY,SITE_STATE_ABBR,SITE_ZIP_CD,"
                 "HCC_TYP_DESC,HCC_STATUS_DESC,HCC_LOC_SETTING_DESC,"
                 "TOT_OPER_HR_PER_WEEK,SITE_POP_TYP_DESC,RURAL_IND,"
                 "MHC_SUB_PROG_IND,SBHC_SUB_PROG_IND,PHPC_SUB_PROG_IND,HCH_SUB_PROG_IND")
    where = f"SITE_STATE_ABBR='{STATE_ABBR}'"
    expected = arcgis_count(HCS, where)
    hc = arcgis_points(HCS, where, HC_FIELDS)
    if len(hc):
        hc["site_kind"] = "Health Center site"
    said = f"{expected:,}" if expected is not None else "n/a"
    print(f"  health center sites in {STATE_ABBR}: service said {said}, got {len(hc):,}")
    check("health center sites: got the rows the service said exist",
          len(hc) > 0 if expected is None else len(hc) == expected,
          f"{len(hc):,} rows; no count available" if expected is None
          else f"{len(hc):,} of {expected:,}")

    leaked = [c for c in hc.columns if "CONTACT" in c.upper()]
    check("no personal contact columns were ingested", not leaked,
          f"found {leaked}" if leaked else "outFields allow-list held")

    if "TOT_OPER_HR_PER_WEEK" in hc.columns:
        hrs = pd.to_numeric(hc["TOT_OPER_HR_PER_WEEK"], errors="coerce")
        print(f"  weekly opening hours: {int(hrs.notna().sum()):,} of {len(hc):,} recorded; "
              f"median {hrs.median():.0f}, min {hrs.min():.0f}, max {hrs.max():.0f}")

    hc = hc.rename(columns={"SITE_NM": "FACILITY_NM", "SITE_ADDRESS": "CMS_PROVIDER_ADDRESS",
                            "SITE_CITY": "CMS_PROVIDER_CITY",
                            "SITE_STATE_ABBR": "CMS_PROVIDER_STATE_ABBR",
                            "SITE_ZIP_CD": "CMS_PROVIDER_ZIP_CD"})
    frames.append(hc)

In [ ]:
with step("place care sites into your county"):
    sites = pd.concat(frames, ignore_index=True, sort=False)
    before = len(sites)
    sites = sites[sites["latitude"].notna() & sites["longitude"].notna()].copy()
    note("every site has coordinates", before - len(sites), before,
         f"{before - len(sites):,} of {before:,} sites had no point geometry")

    # Range-check per jurisdiction, never nationally. A national check passes while an
    # entire state sits in the wrong hemisphere.
    bad = sites[(sites["latitude"] < 17) | (sites["latitude"] > 72) |
                (sites["longitude"] > -60) | (sites["longitude"] < -180)]
    note("coordinates are inside North America", len(bad), len(sites),
         f"{len(bad):,} sites outside the expected box", rows=bad)

    load_df(sites, "care_sites_state")

    # The county assignment is a spatial join, because none of these layers publishes a
    # usable county code. Cheap, exact, and it produces a join the source data cannot.
    sql = f"""
    CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.care_sites` AS
    SELECT s.*, c.geo_id AS county_geo_id, c.county_name
    FROM `{PROJECT}.{DATASET}.care_sites_state` s
    JOIN `bigquery-public-data.geo_us_boundaries.counties` c
      ON ST_CONTAINS(c.county_geom, ST_GEOGPOINT(s.longitude, s.latitude))
    WHERE c.geo_id = '{COUNTY_FIPS}'
    """
    bq.query(sql).result()
    n_sites = bq.get_table(f"{PROJECT}.{DATASET}.care_sites").num_rows
    print(f"  care sites inside {COUNTY_NAME} County: {n_sites:,} of {len(sites):,} statewide")
    by_kind = bq.query(f"""SELECT site_kind, COUNT(*) n
                           FROM `{PROJECT}.{DATASET}.care_sites`
                           GROUP BY 1 ORDER BY 2 DESC""").to_dataframe()
    display(by_kind)

### HRSA shortage areas—and the filter that does nothing

The federal government already has an opinion about where there are not enough providers:
**Health Professional Shortage Areas**. Each carries an `HPSA_SCORE` from 0 to 25, and that score
is more useful than it looks, because **it already contains a travel-time term**. HRSA's
methodology allocates up to 5 of the 25 points to travel time to the nearest source of care.

That matters because there is **no licence-clean federal dataset of travel time to health care**,
and the mapping tool this event gives you cannot compute a route or a distance. `HPSA_SCORE` is
the closest thing to a travel-burden measure we can legitimately use, it is federally computed,
and it costs nothing extra.

### 🔴 There are two HPSA layers and they mean opposite things

This is the easiest serious mistake in the whole dataset, and we made it before you could.

| Layer | What a row is | What it tells you |
|---|---|---|
| **11—component polygons** | A geographic or population shortage **area** | *This neighbourhood is underserved.* **A siting signal.** |
| **9—points** | A shortage **facility** | *This clinic serves an underserved population.* **A marker of where care already is.** |

Every FQHC is designated automatically, just for being an FQHC. So layer 9 for Cook County
includes `ACCESS COMMUNITY HEALTH NETWORK`, `ERIE FAMILY HEALTH CENTER`, several
`Federally Qualified Health Center Look A Like` sites, an `Indian Health Service` clinic—and
**`MCC-Chicago`, a federal jail, designated as a `Correctional Facility` with a score of 3.**

Now look at the county table in Section 3. Travis shows **0 areas and 2 facilities**; Harris
shows **0 and 15**. A team that reads "Travis has 2 HPSAs" as two shortage areas has the meaning
exactly backwards, and a team that merges the layers to fill the Texas gap will be siting a
mobile health clinic partly on the location of a prison.

**We load both, in separate tables, labelled.** `shortage_areas` is the siting signal;
`shortage_facilities` is supply. Do not `UNION` them.

The two layers also **spell the county key differently**—`STATE_COUNTY_FIPS_CD` on layer 11,
**`CMN_STATE_COUNTY_FIPS_CD`** on layer 9, which has 72 fields to layer 11's 83. Query layer 9
with the wrong one and ArcGIS returns **HTTP 200 with an error object in the body**, which any
helper reading `.get("count", 0)` will report as a count of zero. That is how we spent an
afternoon believing Houston had no shortage designations.

### Two more traps, and we walked into the first one ourselves

**`HPSA_WITHDRAWAL_DT IS NULL` matches every single row.** It is the obvious filter, it looks
like it does something, and all 25,490 rows nationally pass it. The column is empty. The real
signal is `HPSA_STATUS_DESC`, which has exactly two values—and **9,957 of 25,490 designations,
39% of the file, are `Proposed For Withdrawal`.** In Cook County it is worse: **498 designated
out of 934, so 47% are on their way out.** Skip the filter and you nearly double your shortage
areas with ones that are being withdrawn.

**`GEO_ID` is mixed-grain.** Some rows carry an eleven-character tract ID, some carry a
five-character county ID: `['48141010337', '48141010336', '48337']`. Join it to tract FIPS
without checking length and a third of the layer silently disappears. We use
`STATE_COUNTY_FIPS_CD` instead and keep `GEO_ID` alongside it, labelled by its own length.

In [ ]:
HPSA = f"{HRSA}/Shortage/HealthProfessionalShortageAreas_FS/MapServer/11"   # primary care

with step("pull HPSA shortage areas"):
    where = (f"STATE_COUNTY_FIPS_CD='{COUNTY_FIPS}' "
             f"AND HPSA_STATUS_DESC='Designated'")
    fields = ("HPSA_NM,HPSA_SCORE,HPSA_STATUS_DESC,HPSA_TYP_DESC,HPSA_DESIGNATION_POP,"
              "HPSA_ESTIMATED_UNDERSERVED_POP,HPSA_ESTIMATED_SERVED_POP,HPSA_POVERTY,"
              "HPSA_FORMAL_RATIO,GEO_ID,STATE_COUNTY_FIPS_CD,COUNTY_NM,RURAL_STATUS_DESC,"
              "HPSA_DESIGNATION_DT")
    n_designated = arcgis_count(HPSA, where)
    n_all = arcgis_count(HPSA, f"STATE_COUNTY_FIPS_CD='{COUNTY_FIPS}'")
    if n_all is None or n_designated is None:
        print(f"  {COUNTY_NAME} County: the service would not report a count; "
              f"pulling rows anyway and counting them ourselves")
        n_designated = n_all = None
    else:
        print(f"  {COUNTY_NAME} County: {n_all:,} designations, "
              f"{n_designated:,} still Designated")
        note("designations in this county are not proposed for withdrawal",
             n_all - n_designated, n_all,
             f"{n_all - n_designated:,} of {n_all:,} are Proposed For Withdrawal",
             limit=1.01)

    if n_designated is None or n_designated:
        hp = arcgis_all(HPSA, where, fields)
        if not len(hp):
            print("  the pull came back empty — no designated shortage AREA here")
            load_df(pd.DataFrame(columns=["HPSA_NM", "HPSA_SCORE"]), "shortage_areas")
            raise SystemExit if False else None
        hp["HPSA_SCORE"] = pd.to_numeric(hp["HPSA_SCORE"], errors="coerce")
        hp["geo_id_len"] = hp["GEO_ID"].astype("string").str.len()
        print(f"  GEO_ID lengths present: {sorted(hp['geo_id_len'].dropna().unique().tolist())}"
              f"  (11 = tract, 5 = county)")
        print(f"  HPSA_SCORE: {int(hp['HPSA_SCORE'].notna().sum())} of {len(hp)} populated, "
              f"range {hp['HPSA_SCORE'].min():.0f}-{hp['HPSA_SCORE'].max():.0f}")
        load_df(hp, "shortage_areas")
    else:
        print("  No designated primary-care shortage AREA in this county. That is a finding,")
        print("  not an error — the federal government has not designated one here, and your")
        print("  agent should be able to say so rather than reporting silence as safety.")
        load_df(pd.DataFrame(columns=["HPSA_NM", "HPSA_SCORE"]), "shortage_areas")

with step("pull HPSA shortage facilities — a DIFFERENT thing"):
    # Layer 9, and note the county key is spelled differently here: CMN_STATE_COUNTY_FIPS_CD.
    # Getting it wrong returns HTTP 200 with an error body, which reads as a count of zero.
    HPSA_FAC = f"{HRSA}/Shortage/HealthProfessionalShortageAreas_FS/MapServer/9"
    fac_fields = ("HPSA_NM,HPSA_TYP_DESC,HPSA_POPULATION_TYP_DESC,HPSA_SCORE,HPSA_STATUS_DESC,"
                  "ADDRESS,CITY,ZIP_CD,HPSA_DESIGNATION_POP,RURAL_STATUS_DESC,"
                  "CMN_STATE_COUNTY_FIPS_CD")
    n_fac = arcgis_count(HPSA_FAC, f"CMN_STATE_COUNTY_FIPS_CD='{COUNTY_FIPS}'")
    print(f"  shortage FACILITIES in {COUNTY_NAME} County: "
          f"{n_fac if n_fac is not None else 'count unavailable'}")
    if n_fac is None or n_fac:
        fac = arcgis_points(HPSA_FAC, f"CMN_STATE_COUNTY_FIPS_CD='{COUNTY_FIPS}'", fac_fields)
        fac["HPSA_SCORE"] = pd.to_numeric(fac["HPSA_SCORE"], errors="coerce")
        print("\n  what kind of facility gets designated:")
        for k, v in fac["HPSA_TYP_DESC"].value_counts().items():
            print(f"    {v:>4}  {k}")
        check("shortage facilities are a supply layer, not a demand layer", True,
              "loaded separately from shortage_areas — do not UNION them")
        load_df(fac, "shortage_facilities")
    else:
        load_df(pd.DataFrame(columns=["HPSA_NM", "HPSA_SCORE"]), "shortage_facilities")

## 8. What's happening this week

Everything so far is structural—it changes over years. A health department also wants to know
what respiratory illness is doing *now*, and CDC's **National Syndromic Surveillance Program**
publishes the percentage of emergency-department visits attributable to COVID, influenza and RSV,
updated weekly and currently about nine days behind.

**Then read the next paragraph twice, because it is the single most misleading thing in this
notebook.**

The file has one row per county per week, with a real five-digit county FIPS. It looks
county-level. **It is not.** The numbers are computed for a **Health Service Area**—a group of
counties that share patterns of care-seeking—and then written onto every county in that group.
CDC says so plainly in its own description.

Travis County's Health Service Area is *"Travis (Austin), TX - Williamson, TX"*, and it covers
**Bastrop, Burnet, Lee, Llano, Travis and Williamson**. Austin, population 1.3 million, is
reported with the same respiratory number as Llano County, population twenty thousand.

**An agent that says "COVID visits are rising in this neighbourhood" using this data has made a
claim the data cannot support.** It can say the region. It cannot say the neighbourhood, and it
certainly cannot say the tract. Getting that distinction right in your output is worth more
credit than any amount of polish.

Two smaller traps:

- **The file mixes grains.** It contains state-level rows alongside county rows, marked by
  `trend_source` and by `county = 'All'`. Pull it naively and you have both in one dataframe.
- **One column name is not what the web interface shows you.** The display name is
  `percent_visits_smoothed_influenza`; the actual field is **`percent_visits_smoothed_1`**.
  Querying the display name returns HTTP 400, *"No such column"*. The unsuffixed
  `percent_visits_smoothed` is the *combined* series, not influenza.
- **About a third of counties have no usable trend** in any given week—`Data Unavailable`,
  `Sparse` or `Limited Data`. Those are concentrated in low-population areas, which are the same
  places the wastewater network misses. **The blind spots correlate.** An agent that reports
  "no increase detected" for a county with no data has said something false.

In [ ]:
NSSP = "rdmq-nq56"

with step("pull NSSP emergency department trends"):
    d = soda(NSSP, {"$select": "week_end, geography, county, fips, hsa, hsa_counties, "
                               "trend_source, percent_visits_combined, percent_visits_covid, "
                               "percent_visits_influenza, percent_visits_rsv, "
                               "percent_visits_smoothed, percent_visits_smoothed_covid, "
                               "percent_visits_smoothed_1, percent_visits_smoothed_rsv, "
                               "ed_trends_covid, ed_trends_influenza, ed_trends_rsv",
                    # fips is typed NUMBER in this dataset, so no quotes. Quoting it
                    # returns zero rows rather than an error.
                    "$where": f"fips={int(COUNTY_FIPS)}",
                    "$order": "week_end"}, page=50000)

    if len(d) == 0:
        raise RuntimeError(
            f"NSSP returned nothing for fips={int(COUNTY_FIPS)}. Check the county code; "
            f"if it is right, this county may be one CDC does not publish.")

    # The grain check. State rows and county rows live in the same file.
    print(f"  rows: {len(d):,}")
    print(f"  trend_source values: {d['trend_source'].value_counts().to_dict()}")
    before = len(d)
    d = d[d["county"] != "All"].copy()
    note("no state-level rows leaked into the county pull", before - len(d), before,
         f"dropped {before - len(d):,} rows where county='All'", limit=1.01)

    for c in [c for c in d.columns if c.startswith("percent_")]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["week_end"] = pd.to_datetime(d["week_end"], errors="coerce")

    hsa = d["hsa"].dropna().unique()
    hsa_counties = d["hsa_counties"].dropna().unique()
    print(f"\n  Health Service Area : {hsa[0] if len(hsa) else 'unknown'}")
    print(f"  counties sharing it : {hsa_counties[0] if len(hsa_counties) else 'unknown'}")
    print(f"  weeks available     : {len(d):,}  "
          f"({d['week_end'].min():%Y-%m-%d} to {d['week_end'].max():%Y-%m-%d})")

    latest = d.sort_values("week_end").tail(1)
    if len(latest):
        r = latest.iloc[0]
        print(f"\n  most recent week {r['week_end']:%Y-%m-%d}:")
        for v in ("covid", "influenza", "rsv"):
            pct = r.get(f"percent_visits_{v}")
            trend = r.get(f"ed_trends_{v}")
            print(f"    {v:<10} {('%.2f%%' % pct) if pd.notna(pct) else '   n/a':>8}   {trend}")

    unusable = int(d["percent_visits_covid"].isna().sum())
    note("this county has a usable COVID trend most weeks", unusable, len(d),
         f"{unusable:,} of {len(d):,} weeks have no value", limit=0.50)

    d = d.rename(columns={"percent_visits_smoothed_1": "percent_visits_smoothed_influenza",
                          "percent_visits_smoothed": "percent_visits_smoothed_combined"})
    n_weeks = load_df(d, "disease_weekly")

## 9. What you have in BigQuery now

Everything above loaded as it went, so there is nothing left to run—this cell just shows you
what is sitting in your project and what each table is keyed on. Print it, screenshot it, and
hand it to whoever is writing queries.

In [ ]:
with step("inventory"):
    q = f"""
    SELECT table_id AS table_name, row_count, ROUND(size_bytes/1048576, 2) AS mb
    FROM `{PROJECT}.{DATASET}.__TABLES__`
    ORDER BY table_id
    """
    inv = bq.query(q).to_dataframe()
    display(inv)
    print("\n  keys:")
    print("    burden_tracts    geo_id      11-char census tract")
    print("    exposure_tracts  geo_id      11-char census tract")
    print("    care_sites       (point)     longitude/latitude, county-filtered")
    print("    shortage_areas       STATE_COUNTY_FIPS_CD + mixed-grain GEO_ID — WHERE NEED IS")
    print("    shortage_facilities  CMN_STATE_COUNTY_FIPS_CD — WHERE CARE IS. Not the same thing")
    print("    care_sites_state     every facility in the state, before the county filter")
    print("    disease_weekly       week_end   one row per week, HSA values on a county key")

## 10. What the data actually says

Here is the question the whole notebook has been building toward, and it is the one a health
department actually asks:

> *How far is each neighbourhood from care it can actually use?*

Not from *a* clinic. Anyone can find the nearest hospital. If you have no insurance, a hospital
is a bill; a **Federally Qualified Health Center** is a sliding-scale appointment. So we measure
two distances for every tract—to the nearest care site of any kind, and to the nearest site that
will see somebody without insurance—and then look at where the two numbers diverge.

Then we cross that against who actually needs it.

In [ ]:
with step("payoff: distance to care you can use"):
    sql = f"""
    WITH t AS (
      SELECT geo_id, total_pop, pct_uninsured, pct_asthma, pct_copd, pct_diabetes,
             pct_no_vehicle, pct_age_65_plus, pct_mobility_difficulty,
             ST_GEOGPOINT(longitude, latitude) AS pt
      FROM `{PROJECT}.{DATASET}.burden_tracts`
      WHERE longitude IS NOT NULL AND latitude IS NOT NULL
    ),
    any_site AS (
      SELECT ST_GEOGPOINT(longitude, latitude) AS pt
      FROM `{PROJECT}.{DATASET}.care_sites`
    ),
    safety_net AS (
      SELECT ST_GEOGPOINT(longitude, latitude) AS pt
      FROM `{PROJECT}.{DATASET}.care_sites`
      WHERE site_kind IN ('FQHC', 'Health Center site')
    ),
    d1 AS (
      SELECT t.geo_id, MIN(ST_DISTANCE(t.pt, s.pt))/1000 AS km_any
      FROM t CROSS JOIN any_site s GROUP BY 1
    ),
    d2 AS (
      SELECT t.geo_id, MIN(ST_DISTANCE(t.pt, s.pt))/1000 AS km_safety_net
      FROM t CROSS JOIN safety_net s GROUP BY 1
    )
    SELECT t.geo_id, t.total_pop, t.pct_uninsured, t.pct_asthma, t.pct_diabetes,
           t.pct_no_vehicle, t.pct_age_65_plus,
           ROUND(d1.km_any, 2)        AS km_to_any_care,
           ROUND(d2.km_safety_net, 2) AS km_to_safety_net,
           ROUND(d2.km_safety_net - d1.km_any, 2) AS extra_km_if_uninsured
    FROM t
    LEFT JOIN d1 USING (geo_id)
    LEFT JOIN d2 USING (geo_id)
    """
    access = bq.query(sql).to_dataframe()
    print(f"  tracts measured: {len(access):,}")

    print()
    spread(access["km_to_any_care"],    "km to ANY care site")
    spread(access["km_to_safety_net"],  "km to a safety-net site")
    spread(access["extra_km_if_uninsured"], "the difference")

    # The population actually affected, which is the number that belongs on a slide.
    far = access[access["km_to_safety_net"] > 5]
    print(f"\n  people living more than 5 km from a safety-net site: "
          f"{int(far['total_pop'].sum()):,} in {len(far):,} tracts")

    bq.query(f"""CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.tract_access` AS {sql}""").result()
    print(f"  wrote {PROJECT}.{DATASET}.tract_access")

In [ ]:
with step("payoff: where need and distance meet"):
    q = f"""
    SELECT geo_id, total_pop,
           ROUND(pct_uninsured,1)  AS uninsured_pct,
           ROUND(pct_asthma,1)     AS asthma_pct,
           ROUND(pct_diabetes,1)   AS diabetes_pct,
           ROUND(pct_no_vehicle,1) AS no_vehicle_pct,
           km_to_any_care, km_to_safety_net
    FROM `{PROJECT}.{DATASET}.tract_access`
    WHERE pct_uninsured   > (SELECT APPROX_QUANTILES(pct_uninsured, 10)[OFFSET(7)]
                             FROM `{PROJECT}.{DATASET}.tract_access`)
      AND pct_no_vehicle  > (SELECT APPROX_QUANTILES(pct_no_vehicle, 10)[OFFSET(6)]
                             FROM `{PROJECT}.{DATASET}.tract_access`)
    ORDER BY km_to_safety_net DESC
    LIMIT 10
    """
    top = bq.query(q).to_dataframe()
    print("  Top decile uninsured AND top third with no vehicle, "
          "ordered by distance to a safety-net site:\n")
    display(top)
    if len(top):
        r = top.iloc[0]
        print(f"  Tract {r.geo_id}: {int(r.total_pop):,} people, "
              f"{r.uninsured_pct}% uninsured, {r.no_vehicle_pct}% with no vehicle, "
              f"{r.km_to_safety_net} km from a clinic that will see them.")

### Sit with that table for a second

Those are neighbourhoods where a large share of working-age adults have no insurance, a large
share of households have no car, and the nearest clinic that will treat somebody without
insurance is several kilometres away.

**A car makes five kilometres nothing. No car makes five kilometres a day off work**, two bus
transfers, and childcare. That is the gap a mobile clinic exists to close, and it is the
strongest single argument your demo can make.

### The warning that comes with those numbers

**Straight-line distance is not travel time.** We are measuring across a map, not along roads.
A tract two kilometres from a clinic on the far side of a river or an interstate is not two
kilometres from that clinic. Say so in your demo before a judge says it for you—and know that
there is no licence-clean federal dataset of travel time to health care, which is precisely why
`HPSA_SCORE` is valuable: HRSA has already folded a travel-time term into it.

**Our facility list is federal only.** County clinics, free clinics, charitable providers and
pharmacy minute-clinics are not in these files because no licence-clean national list of them
exists. Every distance above is therefore an *upper bound*—the real nearest option may be closer
and we cannot see it. **Do not report these as "the nearest care available."** Report them as
"the nearest federally-recognised site," which is what they are.

**And the ranking is only as good as the estimate underneath it.** PLACES values are modelled,
not counted. Two tracts a percentage point apart are, quite possibly, the same tract.

## 11. What we deliberately left out, and why

Every one of these was a decision. If a later session "fixes" one, this is the record of why it
was not broken.

| Left out | Why |
|---|---|
| **SVI Theme 3 and `RPL_THEMES`** | Theme 3 is racial and ethnic minority status. Prohibited as a model input, required as a post-hoc audit. `RPL_THEMES` bakes it in with no column to drop |
| **`FQHC_ADMIN_CONTACT_NM` / `_EMAIL` / `_PHONE_NUM`** | Named individuals. Automatic rejection at this event, and the only reliable defence is never requesting the columns |
| **The four RHC `*_FTE_CT` capacity columns** | Zero of 387 populated. A decoy—see Section 7 |
| **Average PM2.5 as a tract ranking** | 12 km resolution wearing an 11-digit tract ID. Section 2 |
| **Ozone** | Different years, different tract set, numeric keys. A good add-on, a bad default |
| **CDC WONDER** | No sub-national API at all—*"Only national data are available for query by the API"*—plus a data use agreement restricting derived publication, and mortality is a lagging indicator for a preventive programme |
| **Pharmacy locations** | No federal file exists without individual-level PII. NPPES ships practitioners' names and often home addresses; the CMS pharmacy file has NPIs and no addresses. Naming this exclusion is a stronger answer than quietly using NPPES |
| **Mobile clinic registries** | The one national candidate has no licence, no bulk download, and asserts IP over its contents |
| **Wastewater surveillance** | Real, current and licence-clean—but sewersheds span multiple counties and two thirds of US counties have no site. A corroborating signal, never a base layer |
| **Anything patient-level** | This agent plans where a van goes. It does not screen, triage, or advise anybody about their health |

**Race is not in your model. It belongs in your audit.** When your agent produces a ranked list
of neighbourhoods, join it back to the demographics afterwards and ask whether the recommendation
lands disproportionately on any group. If it does, that is a finding to report, not a bug to
hide—and reporting it will earn you more credit than a clean-looking model nobody checked.

## 12. Validate before you build

Everything below queries the **loaded BigQuery tables**, not the dataframes in memory. That
distinction is not pedantry: a previous challenge in this pack ran twenty-three checks against
whatever happened to be sitting in the dataset from a previous run and reported a clean pass for
nine cities while grading a stale copy of a tenth.

Remember the two verdicts. `check()` is about our pipeline and is fatal. `note()` is about the
publisher's data—it warns, prints the offending rows, and only escalates when the problem is
systemic rather than untidy.

*The errors that hurt you are the ones that don't raise.*

In [ ]:
with step("validation"):
    def scalar(sql):
        return list(bq.query(sql).result())[0][0]

    T = f"{PROJECT}.{DATASET}"

    # --- our pipeline did its job
    n_b = scalar(f"SELECT COUNT(*) FROM `{T}.burden_tracts`")
    check("burden_tracts is not empty", n_b > 0, f"{n_b:,} tracts")

    n_bad_id = scalar(f"SELECT COUNTIF(LENGTH(geo_id) != 11) FROM `{T}.burden_tracts`")
    check("every tract id is 11 characters", n_bad_id == 0, f"{n_bad_id:,} wrong length")

    n_pref = scalar(f"SELECT COUNTIF(NOT STARTS_WITH(geo_id, '{COUNTY_FIPS}')) "
                    f"FROM `{T}.burden_tracts`")
    check("every tract is in the county you chose", n_pref == 0,
          f"{n_pref:,} tracts carry another county prefix")

    n_dupe = scalar(f"SELECT COUNT(*) - COUNT(DISTINCT geo_id) FROM `{T}.burden_tracts`")
    check("one row per tract", n_dupe == 0, f"{n_dupe:,} duplicate tract ids")

    n_coord = scalar(f"SELECT COUNTIF(latitude IS NULL OR longitude IS NULL) "
                     f"FROM `{T}.burden_tracts`")
    check("every tract has a centroid", n_coord == 0, f"{n_coord:,} without coordinates")

    lat_ok = scalar(f"SELECT COUNTIF(latitude < 17 OR latitude > 72) FROM `{T}.burden_tracts`")
    check("tract latitudes are degrees, not metres or longitudes", lat_ok == 0,
          f"{lat_ok:,} outside 17-72 degrees")

    n_x = scalar(f"SELECT COUNT(*) FROM `{T}.exposure_tracts`")
    check("exposure_tracts is not empty", n_x > 0, f"{n_x:,} tracts")

    joined = scalar(f"""SELECT COUNT(*) FROM `{T}.burden_tracts` b
                        JOIN `{T}.exposure_tracts` e USING (geo_id)""")
    check("burden and exposure actually join", joined > 0.8 * min(n_b, n_x),
          f"{joined:,} of {min(n_b, n_x):,} possible — ratio {joined/min(n_b,n_x):.0%}")

    n_sites = scalar(f"SELECT COUNT(*) FROM `{T}.care_sites`")
    check("at least one care site in the county", n_sites > 0, f"{n_sites:,} sites")

    n_acc = scalar(f"SELECT COUNTIF(km_to_any_care IS NULL) FROM `{T}.tract_access`")
    check("every tract got a distance", n_acc == 0, f"{n_acc:,} tracts with no distance")

    n_w = scalar(f"SELECT COUNT(*) FROM `{T}.disease_weekly`")
    check("disease_weekly is not empty", n_w > 0, f"{n_w:,} weeks")

    n_all = scalar(f"SELECT COUNTIF(county = 'All') FROM `{T}.disease_weekly`")
    check("no state-level rows in the county disease table", n_all == 0,
          f"{n_all:,} rows with county='All'")

    # --- what the publisher handed us
    imposs = bq.query(f"""SELECT geo_id, pct_uninsured, pct_asthma FROM `{T}.burden_tracts`
                          WHERE pct_uninsured > 100 OR pct_uninsured < 0
                             OR pct_asthma > 100 OR pct_asthma < 0""").to_dataframe()
    note("prevalences are between 0 and 100", len(imposs), n_b,
         f"{len(imposs):,} impossible percentages", rows=imposs)

    nulls = scalar(f"SELECT COUNTIF(pct_uninsured IS NULL) FROM `{T}.burden_tracts`")
    note("every tract has an uninsured estimate", nulls, n_b,
         f"{nulls:,} of {n_b:,} tracts have no PLACES estimate", limit=0.05)

    zeropop = bq.query(f"""SELECT geo_id, total_pop FROM `{T}.burden_tracts`
                           WHERE total_pop = 0 OR total_pop IS NULL""").to_dataframe()
    note("every tract has residents", len(zeropop), n_b,
         f"{len(zeropop):,} tracts report zero or unknown population", limit=0.05,
         rows=zeropop)

    faraway = bq.query(f"""SELECT geo_id, km_to_safety_net FROM `{T}.tract_access`
                           WHERE km_to_safety_net > 100""").to_dataframe()
    note("safety-net distances are plausible", len(faraway),
         scalar(f"SELECT COUNT(*) FROM `{T}.tract_access`"),
         f"{len(faraway):,} tracts more than 100 km from a safety-net site", rows=faraway)

    n_hp = scalar(f"SELECT COUNT(*) FROM `{T}.shortage_areas`")
    note("this county has a designated shortage area", 0 if n_hp else 1, 0,
         f"{n_hp:,} designated primary-care HPSA components")

    # --- the tally
    df = pd.DataFrame(CHECKS)
    print("\n" + "=" * 66)
    display(df)
    tally = df["verdict"].value_counts().to_dict()
    print(f"  passed {tally.get('PASS',0)}   warned {tally.get('WARN',0)}   "
          f"failed {tally.get('FAIL',0)}")
    if tally.get("FAIL"):
        raise RuntimeError(
            f"{tally['FAIL']} check(s) failed — the tables above are not trustworthy. "
            f"Scroll up: each failure printed the rows that caused it.")

## 13. Framing your agent—read this before you write any code

This notebook stops here on purpose. The agent is yours and the design decisions in it are what
you are judged on. But your differentiator is an **architecture**, not a query, which makes this
challenge different from the other four: **you cannot bolt it on at the end.** Decide the shape
in the first ten minutes.

### What makes a problem genuinely need more than one agent

One agent with several tools is enough when the job is: read the request, pick a tool, answer.
Several agents earn their keep when one of these is structurally true—and on this challenge,
three of them are.

**1. Two of your jobs need contradictory instructions.** A clinical-safety reviewer whose whole
prompt is *hedge, cite, refuse if unsure* cannot share a system prompt with an outreach writer
whose prompt is *warm, plain language, sixth-grade reading level, Spanish and English*. Put both
in one agent and you get hedged flyers and chatty triage notes. Try it—it takes two minutes and
it is more convincing than this paragraph.

**2. The equity audit is structurally a second agent, and this is the strong one.** You are
forbidden from using race as a model input and required to audit your output against it. That
means your system must contain a component that sees data the planner was denied. It also must
not share the planner's context, because an agent asked to produce a plan and critique it in the
same breath will defend the plan. **The rule you were given forces the architecture.** Say that
out loud in your demo.

**3. Your evidence gathering is embarrassingly parallel.** Burden, access and exposure are three
independent lookups over three tables. Run them concurrently and the wall clock is one hop
instead of three.

### The one that has evaporated, and you should know why

On older models a built-in tool could not share an agent with a function tool of your own, which
forced a sub-agent wrapper. **On Gemini 3.x that restriction is gone.** Do not cite it as your
reason for going multi-agent; a judge who knows will not be impressed.

### The API, verified in this environment on 2026-08-10

ADK's `SequentialAgent`, `ParallelAgent` and `LoopAgent` still work, but **all three are
deprecated** in favour of the graph `Workflow`:

```
DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be
removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
```

The shape that works, and that we ran end to end in a lab project exactly like yours:

```python
from pydantic import BaseModel
from google.adk import Workflow, Event
from google.adk.workflow import node, START
from google.adk.agents import Context, LlmAgent

class PlanRequest(BaseModel):
    county: str

@node
async def gather_burden(ctx: Context):
    ...                                     # a plain function. No model call.
    return Event(message="burden gathered")

plan_graph = Workflow(
    name="clinic_plan",
    description="Plans mobile clinic stops for a county.",   # REQUIRED to use as a tool
    input_schema=PlanRequest,                                # ALSO REQUIRED
    timeout=120.0,                                           # SEE BELOW. NOT OPTIONAL.
    edges=[(START, gather_burden, ...)],
)

root_agent = LlmAgent(name="coordinator", model="gemini-3.6-flash", tools=[plan_graph])
```

**Five things that will cost you time if you meet them by accident:**

**A graph cycle with no `timeout` runs forever.** `Workflow` has no iteration-cap field of any
kind, and `timeout` defaults to `None`. We built a deliberately infinite plan-audit loop and it
ran until an external backstop killed it. With `timeout=5.0` it raised `NodeTimeoutError`
cleanly. **If your graph contains a cycle, set `timeout`.** `LoopAgent` at least has
`max_iterations`; a graph does not.

**Both `description` and `input_schema` are required** to use a `Workflow` as a tool, in that
order of complaint. Without a description: *"must have a description to be wrapped as a tool."*
With one but no schema: *"NodeTool requires an explicit Pydantic input_schema."*

**A schema'd workflow takes a JSON string, not a dict.** `run_debug('{"county": "Travis"}')`
works. A dict or a model instance gets iterated into its keys and fails validation, because the
runner is typed for `str | list[str]`.

**Parallel branches share one state dict and will clobber each other.** Two branches both writing
`output_key="shared"` leave one value. The isolation is on conversation history, not state.
**Every parallel branch needs a distinct `output_key`.**

**A missing `{key}` in an instruction is a hard crash**, not a silent blank:
`KeyError: 'Context variable not found'`. Use `{key?}` if you want it optional. This is good
news—but it means you must run every agent once after wiring it.

### How to see it working, which is most of debugging

```python
from google.adk.plugins import LoggingPlugin
runner = InMemoryRunner(agent=root_agent, plugins=[LoggingPlugin()])
await runner.run_debug("Plan next month's clinic stops.", verbose=True)
```

One line, and you get every agent transition, every model request, every tool call, and **token
usage per call**—which is how you find out whether your six-agent loop is affordable before you
run it fifty times. In `adk web`, the **State** tab plus the event graph lets you click an event
and see which node produced it.

Two environment notes: in a notebook use a bare `await`, because `asyncio.run()` raises inside a
cell. And `adk web` defaults to port 8000 on 127.0.0.1, so in Cloud Shell run
`adk web --host 0.0.0.0 --port 8080` and use Web Preview.

### Where the good version separates from the merely working one

Every team gets these same five tables. What separates you:

- It **says which scale each claim came from**—tract, 12 km grid, facility point, or a
  six-county Health Service Area—instead of blending them into one confident sentence.
- It **counts what it cannot see.** A third of counties have no usable disease trend, eleven
  states have no transportation measure, and "no signal" is not "no problem."
- It **audits its own output** against the demographics it was forbidden to rank on, and reports
  what it finds even when the finding is awkward.
- It knows that **five kilometres means nothing with a car and a day off work without one.**
- It is honest when it does not know, in a domain where a confident wrong answer is the worst
  possible output.

**The four questions a judge will ask, so you may as well be ready:**

1. *Delete any one agent. Does the answer get worse?* If not, that agent was decoration.
2. *Show me a key in `session.state` written by one agent and read by another.* If there isn't
   one, you have a single agent and some function calls.
3. *Show me two agents whose instructions genuinely conflict.* Near-identical prompts are one
   agent wearing hats.
4. *Show me the point where your system rejected its own first answer.*

## 14. Appendix—the diagnostic block

One cell, one paste. If you ask a coach for help, run this first and give them the output—it
answers most of the questions they would otherwise have to ask you.

In [ ]:
print("=" * 74)
print(f"A4I 2026 — CHALLENGE 5 DIAGNOSTIC   county={COUNTY_NAME} ({COUNTY_FIPS}) {STATE_ABBR}")
print("=" * 74)
print(f"project        : {PROJECT}")
print(f"dataset        : {DATASET}   location={LOCATION}")
print(f"python         : {sys.version.split()[0]}   pandas={pd.__version__}")
print(f"notebook       : c5_01_load_explore")

print("\n-- tables " + "-" * 63)
try:
    inv = bq.query(f"""SELECT table_id, row_count
                       FROM `{PROJECT}.{DATASET}.__TABLES__` ORDER BY table_id""").to_dataframe()
    for _, r in inv.iterrows():
        print(f"  {r['table_id']:<22} {int(r['row_count']):>9,} rows")
except Exception as e:
    print(f"  could not list tables: {e}")

print("\n-- headline numbers " + "-" * 53)
try:
    T = f"{PROJECT}.{DATASET}"
    h = bq.query(f"""
      SELECT COUNT(*) AS tracts, SUM(total_pop) AS population,
             ROUND(AVG(pct_uninsured),1) AS mean_uninsured,
             ROUND(MIN(pct_uninsured),1) AS min_uninsured,
             ROUND(MAX(pct_uninsured),1) AS max_uninsured,
             ROUND(AVG(pct_asthma),1) AS mean_asthma
      FROM `{T}.burden_tracts`""").to_dataframe().iloc[0]
    # Bracket access, not attribute access. `h.pop` resolves to Series.pop — the METHOD —
    # and you get a TypeError three lines later naming int() rather than the real problem.
    print(f"  tracts {int(h['tracts']):,}   population {int(h['population']):,}")
    print(f"  uninsured 18-64: mean {h['mean_uninsured']}%  "
          f"range {h['min_uninsured']}-{h['max_uninsured']}%")
    print(f"  asthma: mean {h['mean_asthma']}%")
    a = bq.query(f"""
      SELECT ROUND(APPROX_QUANTILES(km_to_any_care,2)[OFFSET(1)],2) med_any,
             ROUND(APPROX_QUANTILES(km_to_safety_net,2)[OFFSET(1)],2) med_net,
             ROUND(MAX(km_to_safety_net),2) worst_net
      FROM `{T}.tract_access`""").to_dataframe().iloc[0]
    print(f"  median km to any care {a.med_any}   to safety net {a.med_net}   "
          f"worst {a.worst_net}")
    e = bq.query(f"""SELECT ROUND(MIN(mean_pm),2) AS lo, ROUND(MAX(mean_pm),2) AS hi
                     FROM `{T}.exposure_tracts`""").to_dataframe().iloc[0]
    _lo, _hi = float(e['lo']), float(e['hi'])
    print(f"  tract mean PM2.5 range {_lo} - {_hi} ug/m3  "
          f"(ratio {round(_hi/_lo, 3) if _lo else 'n/a'})")
    w = bq.query(f"""SELECT MAX(week_end) AS mx, COUNT(*) AS n
                     FROM `{T}.disease_weekly`""").to_dataframe().iloc[0]
    print(f"  disease_weekly: {int(w['n']):,} weeks, latest {w['mx']}")
except Exception as ex:
    print(f"  could not compute: {type(ex).__name__}: {ex}")

print("\n-- validation " + "-" * 59)
_df = pd.DataFrame(CHECKS)
if len(_df):
    _t = _df["verdict"].value_counts().to_dict()
    print(f"  passed {_t.get('PASS',0)}   warned {_t.get('WARN',0)}   failed {_t.get('FAIL',0)}")
    for _, r in _df[_df["verdict"] != "PASS"].iterrows():
        print(f"  {r['verdict']}  {r['check']} — {r['detail']}")
else:
    print("  no checks recorded — did section 12 run?")

print("\n-- timings " + "-" * 62)
for k, v in STEPS.items():
    print(f"  {v:>7.1f}s  {k}")
print(f"  {sum(STEPS.values()):>7.1f}s  TOTAL")
print("=" * 74)